<!-- NB_DOC_INTRO_V1 -->
# NLP — Transformers (HuggingFace)

> 📚 **Doc thematique** : [docs/05_NLP.md](docs/05_NLP.md) (NLP)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Transformers** = architecture qui domine le NLP depuis 2018 (BERT) → 2025 (GPT, Llama, Claude). Ce notebook execute les **patterns essentiels** avec HuggingFace `transformers` : tokenization, modeles pre-entraines, fine-tuning (concepts), `pipeline` (zero-shot, NER, sentiment, summarization).

## Plan

1. Setup + concepts (Tokenizer, Model, Pipeline)
2. Tokenization (BPE, WordPiece, SentencePiece)
3. Modeles pre-entraines (BERT, RoBERTa, T5, GPT)
4. Pipeline (sentiment, NER, summarization, zero-shot)
5. Embeddings de texte (last_hidden_state, mean pooling)
6. Fine-tuning (concepts + code de reference)
7. Inference optimisee (quantization, batching)
8. Pieges + References


In [ ]:
try:
    import transformers
    from transformers import AutoTokenizer, AutoModel, pipeline
    TR_OK = True
    print(f"transformers : {transformers.__version__}")
except ImportError:
    TR_OK = False
    print("transformers pas installe : uv add --group nlp transformers")

import numpy as np
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Concepts cles

| Composant | Role |
|---|---|
| **Tokenizer** | Decoupe texte en tokens (sub-words), genere `input_ids` |
| **Model** | Reseau neuronal (BERT-base = 110M params, BERT-large = 340M) |
| **Pipeline** | Wrapper haut niveau : `pipeline("sentiment-analysis")` |
| **Trainer** | Boucle d'entrainement standard (HF api) |
| **Hub** | Repository de modeles : https://huggingface.co/models |

### Variantes de tokenization

| Algo | Modeles utilisateurs |
|---|---|
| **WordPiece** | BERT, DistilBERT |
| **BPE** (Byte Pair Encoding) | GPT, Llama |
| **SentencePiece** | T5, Llama, multilingual |
| **Byte-level BPE** | GPT-2, RoBERTa |

Toutes : decoupage sub-word → gere les mots inconnus.


## 2. Tokenization

In [ ]:
if TR_OK:
    # AutoTokenizer trouve le bon tokenizer pour le modele
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

    text = "Hugging Face Transformers is amazing!"
    encoded = tokenizer(text, return_tensors="pt")

    print(f"Input text  : {text}")
    print(f"Token IDs   : {encoded['input_ids'][0].tolist()}")
    print(f"Tokens      : {tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])}")
    print(f"Decoded back: {tokenizer.decode(encoded['input_ids'][0])}")

    # Batch + padding
    batch = ["Short text", "A much longer text that needs to be truncated or padded."]
    enc = tokenizer(batch, padding=True, truncation=True, max_length=20, return_tensors="pt")
    print(f"\nBatch shape : {enc['input_ids'].shape}")
    print(f"Attention mask :\n{enc['attention_mask']}")
else:
    print("SKIP — transformers indispo")

## 3. Pipeline (le plus simple)

In [ ]:
if TR_OK:
    # Sentiment analysis (auto-DL un modele DistilBERT fine-tuned SST-2)
    sentiment = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
    results = sentiment([
        "I love this product, it's amazing!",
        "This is the worst experience ever.",
        "It's okay, nothing special.",
    ])
    for text, r in zip(["good", "bad", "neutral"], results):
        print(f"{text:8s} : {r['label']} (score={r['score']:.3f})")
else:
    print("SKIP")

In [ ]:
# Zero-shot classification (no fine-tuning required)
if TR_OK:
    zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
    result = zs(
        "I'm planning a trip to Paris next month.",
        candidate_labels=["travel", "food", "sport", "politics"],
    )
    print(f"Texte : 'I'm planning a trip to Paris next month.'")
    for lbl, score in zip(result["labels"], result["scores"]):
        print(f"  {lbl:10s} : {score:.3f}")
else:
    print("SKIP")

## 4. NER avec pipeline

In [ ]:
if TR_OK:
    ner = pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english",
                    aggregation_strategy="simple")
    entities = ner("Sundar Pichai is the CEO of Google, headquartered in Mountain View.")
    for ent in entities:
        print(f"  {ent['word']:25s} | {ent['entity_group']} (score={ent['score']:.3f})")
else:
    print("SKIP")

## 5. Text classification (multi-label / multi-class)

In [ ]:
if TR_OK:
    try:
        # Quelques versions de transformers ne supportent plus 'summarization' via pipeline
        # On utilise text-classification a la place (toujours dispo)
        clf = pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english")
        texts = ["This movie was fantastic!", "Worst film I've seen.", "It was okay."]
        for t, r in zip(texts, clf(texts)):
            print(f"  {t!r:50s} → {r['label']} ({r['score']:.3f})")
    except Exception as e:
        print(f"Pipeline error : {e}")
else:
    print("SKIP")

## 6. Embeddings (mean pooling sur last_hidden_state)

In [ ]:
if TR_OK:
    import torch

    model = AutoModel.from_pretrained("distilbert-base-uncased")
    model.eval()

    texts = [
        "I love machine learning",
        "I hate slow internet",
        "Machine learning is awesome",
    ]
    enc = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        out = model(**enc)

    # Mean pooling sur les tokens non-pad
    mask = enc["attention_mask"].unsqueeze(-1).float()
    embeddings = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
    print(f"Embeddings shape : {embeddings.shape}")

    # Cosine similarity
    from sklearn.metrics.pairwise import cosine_similarity
    sims = cosine_similarity(embeddings.numpy())
    print(f"\nSimilarity matrix :\n{sims.round(3)}")
    print("\n(Phrases 1 et 3 doivent etre similaires — sujet 'ML')")
else:
    print("SKIP")

**Note** : pour des embeddings de phrases de qualite production, utiliser **sentence-transformers** (fine-tunes specifiquement pour la similarite) plutot que la sortie raw d'un BERT.

```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(texts)  # plus precis pour similarite
```

## 7. Fine-tuning (concepts + code de reference)

Le pattern HuggingFace standard pour fine-tuner sur sa propre tache :

```python
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. Load dataset
ds = load_dataset("imdb")

# 2. Tokenize
def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True, max_length=512)
ds_tok = ds.map(tokenize, batched=True)

# 3. Load model (head random pour 2 classes)
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# 4. Training args
args = TrainingArguments(
    output_dir="./out",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,                   # GPU only
    report_to="wandb",           # ou "tensorboard", "mlflow", "none"
)

# 5. Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"].select(range(5000)),    # subset pour rapidite
    eval_dataset=ds_tok["test"].select(range(1000)),
    tokenizer=tokenizer,
)
trainer.train()
trainer.evaluate()
```

Pour **PEFT / LoRA** (fine-tuning leger) : [AI_LLM_Finetuning_PEFT_LoRA.ipynb](./AI_LLM_Finetuning_PEFT_LoRA.ipynb).

## 8. Inference optimisee

### Quantization (8-bit, 4-bit)
```python
from transformers import BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModel.from_pretrained("model_id", quantization_config=bnb, device_map="auto")
```

### Batching
Toujours grouper les inferences (1 forward sur batch >> N forwards seuls).

### ONNX export
Pour deployer hors PyTorch :
```bash
optimum-cli export onnx --model distilbert-base-uncased ./onnx-model
```

## 9. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Pas `model.eval()` en inference | Dropout actif → predictions stochastiques |
| Pas `torch.no_grad()` en eval | Memoire gaspillee |
| `padding=True` sans `max_length` | Sequences tres longues → OOM |
| Charger BERT-large sur CPU pour inference simple | DistilBERT 60% plus rapide, 97% du score |
| Pas grouper en batch | 10x plus lent que necessaire |
| Embeddings raw BERT pour similarite | Sentence-Transformers est specifiquement fine-tune pour ca |
| Fine-tuning sans `eval_strategy` | Pas de detection overfitting |
| Charger 100x le meme modele dans une boucle | Charger 1 fois, garder en memoire |


## References

### Documentation
- HuggingFace docs : https://huggingface.co/docs/transformers/
- HuggingFace course : https://huggingface.co/learn/nlp-course
- Sentence-Transformers : https://www.sbert.net/
- Attention is All You Need (Vaswani et al. 2017) : https://arxiv.org/abs/1706.03762

### Voir aussi
- [NLP_NER.ipynb](NLP_NER.ipynb)
- [NLP_LangChain_RAG.ipynb](NLP_LangChain_RAG.ipynb)
- [AI_LLM_Finetuning_PEFT_LoRA.ipynb](AI_LLM_Finetuning_PEFT_LoRA.ipynb)
- [AI_Local_LLMs.ipynb](AI_Local_LLMs.ipynb)
- [AI_Prompt_Engineering.ipynb](AI_Prompt_Engineering.ipynb)
